In [7]:
from utils_extraction import tokenize, decode
from utils_extraction import is_auto_label_tag
from utils_extraction import extract_few_shot_examples_from_labels
from utils_extraction import select_few_shot 
from utils_extraction import process_labels
from utils_extraction import add_attributes_to_auto_labels, compare_html_allow_auto_labels
from utils_extraction import SUBLABEL_DEFINITIONS_V1, SUBLABEL_DEFINITIONS_V2
from models import GPTAssistant
import json

In [10]:
# ---------- Define Hyperparameters ----------
model_name = "gpt-5.2"

n_few_shot = 20  # Number of few-shot examples to use
fs_mode = "selected"  # "random" or "selected"

#### Define the text to process, and where to save it

In [22]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "1997CanLII16226_ONCA" #"2021QCCA1675" #"1997CanLII16226_ONCA"
anno = "llm"
version = "v1.1"
out_version = "v1.2"
html_path = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2\{filename}_llm_{version}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2"


# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")





   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2\1997CanLII16226_ONCA_llm_v1.1.html


### Process The HTML Content

In [23]:
# ---------- Tokenize html content ----------
tokens = tokenize(html_content)

### Few shot selection

In [24]:
if out_version == "v1.1":
    sublabel_config = {
    "parent":["decision", "legislation", "secondary sources"], # only extract sublabels under these parents
    "already_labeled":[], # do not extract sublabels under these labels
    "new_labels":["title", "fragment"],
    "keep_attributes":["labelname"],
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]
    
if out_version == "v1.2":
    sublabel_config = {
    "parent":["secondary sources"], # only extract sublabels under these parents
    "already_labeled":["title", "fragment"], # do not extract sublabels under these labels
    "new_labels":["source", "authors"],
    "keep_attributes":["labelname"], 
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]

if out_version == "v1.3":
    sublabel_config = {
        "parent":["decision", "legislation"], # only extract sublabels under these parents
        "already_labeled":["title", "fragment", "source", "authors"], # do not extract sublabels under these labels
        "new_labels":["citation"],
        "keep_attributes":["labelname"], 
        "switch_type":True, # manual_label -> auto_label
        "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.8]

# Do not use remove_labels here, as we need the parent labels to identify sublabels : This could  create issues.

In [25]:
if fs_mode == "random" :
    fs_filename = "2019SCC65_annotated_EG_v1_corrected"
    fs_anno = "EG"
    fs_version = "v1"
    fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"
    # Read HTML file
    with open(fs_html_path, 'r', encoding='utf-8') as file:
        fs_html_content = file.read()
    print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")

    # ---------- Tokenize html content ----------
    fs_tokens = tokenize(fs_html_content)

    # ---------- Create few-shot examples ----------

    few_shot_examples = extract_few_shot_examples_from_labels(fs_tokens, 
                                                sublabel_config)


    # Select examples with distributed method: 50% with "source" label, 50% random others
    selected_few_shot_examples = select_few_shot(
        examples=few_shot_examples, 
        n=n_few_shot,
        method="distributed",
        list_of_labels=sublabel_config["new_labels"],
        distribution=distribution
    )
    print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

    print(selected_few_shot_examples)

In [26]:
if fs_mode == "selected":
    
    # Load the selected few-shot examples JSON
    fs_json_path = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json"
    with open(fs_json_path, 'r', encoding='utf-8') as file:
        fs_data = json.load(file)
    print(f"   ✓ Loaded {len(fs_data)} examples from: {fs_json_path}")
    
    # Get all unique source files and print them
    source_files = sorted(list(set([item.get('source_file', 'unknown') for item in fs_data])))
    print(f"\n   Source files in few-shot collection:")
    for sf in source_files:
        count = sum(1 for item in fs_data if item.get('source_file') == sf)
        print(f"      - {sf} ({count} examples)")
    
    # Apply filters:
    # 1. Mask filter: Exclude examples from the same document being annotated (avoid data leakage)
    # 2. Manual filter: Only keep examples where "selected" == true
    current_doc_base = filename
    
    filtered_examples = []
    excluded_same_doc = 0
    excluded_not_selected = 0
    
    for item in fs_data:
        source_file = item.get('source_file', '')
        
        # Filter 1: Check if the current document name appears in the source file (MASK FILTER)
        if current_doc_base in source_file:
            excluded_same_doc += 1
            continue
        
        # Filter 2: Only keep examples with "selected" == true (MANUAL FILTER)
        if not item.get('selected', False):
            excluded_not_selected += 1
            continue
        
        # Extract input/output from the example
        if 'example' in item and 'input' in item['example'] and 'output' in item['example']:
            filtered_examples.append({
                'input': item['example']['input'],
                'output': item['example']['output'],
                'source_file': source_file
            })
    
    print(f"\n   ✓ Filtering results:")
    print(f"      - Excluded (same document): {excluded_same_doc}")
    print(f"      - Excluded (not selected): {excluded_not_selected}")
    print(f"      - Retained: {len(filtered_examples)}")
    
    # Show source files of retained examples
    retained_sources = {}
    for ex in filtered_examples:
        sf = ex['source_file']
        retained_sources[sf] = retained_sources.get(sf, 0) + 1
    
    print(f"\n   ✓ Retained examples come from:")
    for sf, count in sorted(retained_sources.items()):
        print(f"      - {sf}: {count} examples")
    
    # Select the required number of examples
    if len(filtered_examples) > n_few_shot:
        selected_examples_dicts = filtered_examples[:n_few_shot]
    else:
        selected_examples_dicts = filtered_examples
    
    # Simplify examples based on sublabel_config
    def simplify_for_sublabel_extraction(html_text, keep_tags):
        """
        Simplify annotated text for sublabel extraction.
        Keep only specified tags, remove all others (but keep their content).
        
        Args:
            html_text: HTML text with tags
            keep_tags: List of tag names to keep (e.g., ['legislation', 'decision', 'title'])
        
        Example: 
            keep_tags=['legislation', 'title']
            <legislation><title>Act</title>, <citation>123</citation></legislation>
            -> <legislation><title>Act</title>, 123</legislation>
        """
        # Tokenize the HTML text
        tokens = tokenize(html_text)
        
        simplified_tokens = []
        
        for token in tokens:
            # Check if it's an opening tag
            if token.startswith('<') and not token.startswith('</') and token.endswith('>'):
                tag_name = token[1:-1].split()[0]  # Get tag name without attributes
                
                if tag_name in keep_tags:
                    # This is a tag we want to keep
                    simplified_tokens.append(token)
                # Otherwise skip the tag (but content will be kept)
            
            # Check if it's a closing tag
            elif token.startswith('</') and token.endswith('>'):
                tag_name = token[2:-1]
                
                if tag_name in keep_tags:
                    # This is a closing tag we want to keep
                    simplified_tokens.append(token)
                # Otherwise skip the closing tag
            
            else:
                # Regular text content, always keep it
                simplified_tokens.append(token)
        
        # Decode back to string
        return decode(simplified_tokens)
    
    print(f"\n   ✓ Simplifying examples for sublabel extraction (version {out_version})...")
    
    # Determine which tags to keep based on sublabel_config
    # Input: parent tags + already_labeled tags
    # Output: parent tags + already_labeled tags + new_labels
    parent_tags = sublabel_config.get('parent', [])
    already_labeled = sublabel_config.get('already_labeled', [])
    new_labels = sublabel_config.get('new_labels', [])
    
    input_keep_tags = parent_tags + already_labeled
    output_keep_tags = parent_tags + already_labeled + new_labels
    
    print(f"      - Input will keep: {input_keep_tags}")
    print(f"      - Output will keep: {output_keep_tags}")
    
    simplified_examples = []
    for ex in selected_examples_dicts:
        simplified_input = simplify_for_sublabel_extraction(ex['input'], input_keep_tags)
        simplified_output = simplify_for_sublabel_extraction(ex['output'], output_keep_tags)
        simplified_examples.append((simplified_input, simplified_output))
    
    # Convert to list of tuples (input, output)
    selected_few_shot_examples = simplified_examples
    
    print(f"\n   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")
    
    # Print first example as verification (before and after simplification)
    if selected_few_shot_examples and selected_examples_dicts:
        print("\n   First example preview:")
        print("   Input (original):", selected_examples_dicts[0]['input'][:150], "...")
        print("   Input (simplified):", selected_few_shot_examples[0][0][:150], "...")
        print("   Output (original):", selected_examples_dicts[0]['output'][:150], "...")
        print("   Output (simplified):", selected_few_shot_examples[0][1][:150], "...")

   ✓ Loaded 270 examples from: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json

   Source files in few-shot collection:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json (35 examples)
      - few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json (155 examples)
      - few_shot_examples_2019SCC65_annotated_EG_tech_corrected.json (59 examples)
      - few_shot_examples_2021QCCA1675_annotated_EG_tech.json (21 examples)

   ✓ Filtering results:
      - Excluded (same document): 155
      - Excluded (not selected): 94
      - Retained: 21

   ✓ Retained examples come from:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json: 12 examples
      - few_shot_examples_2019SCC65_annotated_EG_tech_corrected.json: 4 examples
      - few_shot_examples_2021QCCA1675_annotated_EG_tech.json: 5 examples

   ✓ Simplifying examples for sublabel extraction (version v1

In [27]:
selected_few_shot_examples

[(' THE\nMINISTER OF NATIONAL REVENUE Respondent Excise\nTax Act - Whether sign bridge assemblies are bridges within the meaning of\nparagraph\xa01(h), Part XII, Schedule III of the Excise Tax Act. DECISION: The appeal is\ndismissed. The word "bridge" must',
  ' THE\nMINISTER OF NATIONAL REVENUE Respondent <title>Excise\nTax Act</title> - Whether sign bridge assemblies are bridges within the meaning of\n<fragment>paragraph\xa01(h), Part XII, Schedule III</fragment> of the <title>Excise Tax Act</title>. DECISION: The appeal is\ndismissed. The word "bridge" must'),
 (' Peter\nC. Engelmann, for the respondent Statutes Cited: Canadian\nInternational Trade Tribunal Act, S.C. 1988, c. 56,subs.54(2) and s. 60; Excise\nTax Act, R.S.C. 1970, c. E-13, paragraph\xa01(h), Part\xa0XII, Schedule\nIII. Cases',
  ' Peter\nC. Engelmann, for the respondent Statutes Cited:  <title>Canadian\nInternational Trade Tribunal Act</title>, S.C. 1988, c. 56,<fragment>subs.54(2)</fragment> and <fragment>s. 60</fra

### Processing

In [19]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [20]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_sublabels_extraction_from_parent_cot.txt"
sublabel_definitions = SUBLABEL_DEFINITIONS_V2
processed_label = process_labels(
    model=model,
    tokens=tokens,
    sublabel_config=sublabel_config,
    few_shot_examples=selected_few_shot_examples,
    prompt_path=prompt_path,
    sublabel_definitions=sublabel_definitions,
    output_dir=output_dir,
    filename=filename,
)


   ✓ Found 595 parent mentions to process
   ✓ Built 1191 token segments (595 to process)


Processing mentions: 100%|██████████| 1191/1191 [17:14<00:00,  1.15it/s]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2\history_1997CanLII16226_ONCA_sublabel.json
   ✓ Processed tokens saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2\processed_sublabels_1997CanLII16226_ONCA.json

   ✓ Sublabel extraction completed:
      - Total mentions: 595
      - Successful: 595
      - Failed: 0


### Post Processing

In [21]:
# Useless verification to check if the tokens are the same after processing (except for the auto labels)
t1 = []
t2 = []
for token in processed_label:
    if not is_auto_label_tag(token) in [1, 2]:
        t1.append(token)


for token in tokens:
    if not is_auto_label_tag(token) in [1, 2]:
        t2.append(token)

assert t1 == t2, "The tokens are different after processing, which should not happen as we are only adding auto_label tags without changing the original tokens."


processed_html = decode(processed_label)

print(f"\HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = add_attributes_to_auto_labels(processed_html)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_{anno}_{out_version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")


\HTML length: 320231
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\v_prompt_2_500_selected_30_gpt5.2_chunk2_subdef2


<>:18: SyntaxWarning: invalid escape sequence '\H'
<>:18: SyntaxWarning: invalid escape sequence '\H'
C:\Users\zakga\AppData\Local\Temp\ipykernel_7248\1416758461.py:18: SyntaxWarning: invalid escape sequence '\H'
  print(f"\HTML length: {len(processed_html)}")
